In [1]:
import os
import pandas as pd
from bs4 import BeautifulSoup
from io import StringIO

In [2]:
SCORE_DIR = "data/scores"

In [3]:
box_scores = os.listdir(SCORE_DIR)

In [4]:
box_scores = [os.path.join(SCORE_DIR,f) for f in box_scores if f.endswith ("html")]

In [5]:
def parse_html(box_score):
    with open(box_score) as f:
        html = f.read()

    soup = BeautifulSoup(html)
    [s.decompose() for s in soup.select("tr.over_header")]
    [s.decompose() for s in soup.select("tr.thread")]
    return soup

In [6]:
def read_line_score(soup):
    line_score = pd.read_html(StringIO(str(soup)), attrs = {"id": "line_score"})[0]
    cols = list(line_score.columns)
    cols[0] = "team"
    cols[-1] = "total"
    line_score.columns = cols

    line_score = line_score[["team","total"]]
    return line_score

In [7]:
def read_stats(soup, team, stat):
    df = pd.read_html(StringIO(str(soup)), attrs = {"id": f"box-{team}-game-basic"}, index_col = 0)[0]
    df = df.apply(pd.to_numeric, errors = "coerce")
    return df

In [8]:
def read_season_info(soup):
    nav = soup.select("#bottom_nav_container")[0]
    hrefs = [a["href"] for a in nav.find_all("a")]
    season = os.path.basename(hrefs[1]).split("_")[0]
    return season

In [11]:
base_cols = None
games = []
failed = []

for i, box_score in enumerate(box_scores):
    try:
        soup = parse_html(box_score)
        line_score = read_line_score(soup)
        teams = list(line_score["team"])

        summaries = []
        for team in teams:
            basic = read_stats(soup, team, "basic")
            advanced = read_stats(soup, team, "advanced")
        
            totals = pd.concat([basic.iloc[-1,:], advanced.iloc[-1,:]])
            totals.index = totals.index.str.lower()
            totals = totals[~totals.index.duplicated(keep="first")]
        
            maxes = pd.concat([basic.iloc[:-1,:].max(), advanced.iloc[:-1,:].max()])
            maxes.index = maxes.index.str.lower() + "_max"
            maxes = maxes[~maxes.index.duplicated(keep="first")]
        
            summary = pd.concat([totals, maxes])
        
            if base_cols is None:
                base_cols = list(summary.index.drop_duplicates(keep="first"))
                base_cols = [b for b in base_cols if "bpm" not in b]
        
            summary = summary.reindex(base_cols)
            summaries.append(summary)

        summary = pd.concat(summaries, axis=1).T

        game = pd.concat([summary, line_score], axis=1)

        game["home"] = [0, 1]
        game_opp = game.iloc[::-1].reset_index(drop=True)
        game_opp.columns += "_opp"

        full_game = pd.concat([game, game_opp], axis=1)
        full_game["season"] = read_season_info(soup)

        full_game["date"] = os.path.basename(box_score)[:8]
        full_game["date"] = pd.to_datetime(full_game["date"], format="%Y%m%d")
        full_game["won"] = full_game["total"] > full_game["total_opp"]

        games.append(full_game)

    except Exception as e:
        failed.append((box_score, str(e)))
        continue

    if len(games) % 100 == 0:
        print(f"{len(games)} / {len(box_scores)}")

print(f"Failed on {len(failed)} files")

100 / 3527
200 / 3527
300 / 3527
400 / 3527
500 / 3527
600 / 3527
700 / 3527
800 / 3527
900 / 3527
1000 / 3527
1100 / 3527
1200 / 3527
1300 / 3527
1400 / 3527
1500 / 3527
1600 / 3527
1700 / 3527
1800 / 3527
1900 / 3527
2000 / 3527
2100 / 3527
2200 / 3527
2300 / 3527
2400 / 3527
2500 / 3527
2600 / 3527
2700 / 3527
2800 / 3527
2900 / 3527
3000 / 3527
3100 / 3527
3200 / 3527
3300 / 3527
3400 / 3527
3500 / 3527
Failed on 21 files


In [13]:
games_df = pd.concat(games, ignore_index = True)

In [14]:
games_df

,mp,fg,fga,fg%,3p,3pa,3p%,ft,fta,ft%,...,pf_max_opp,pts_max_opp,gmsc_max_opp,+/-_max_opp,team_opp,total_opp,home_opp,season,date,won
0,240.0,42.0,91.0,0.462,14.0,39.0,0.359,20.0,23.0,0.870,...,5.0,23.0,20.3,7.0,POR,113,1,2023,2022-11-19,True
1,240.0,39.0,83.0,0.470,13.0,42.0,0.310,22.0,27.0,0.815,...,4.0,29.0,20.2,14.0,UTA,118,0,2023,2022-11-19,False
2,240.0,39.0,92.0,0.424,8.0,33.0,0.242,18.0,19.0,0.947,...,4.0,32.0,28.1,23.0,CHO,115,1,2025,2025-01-07,False
3,240.0,38.0,98.0,0.388,13.0,47.0,0.277,26.0,29.0,0.897,...,5.0,39.0,32.5,15.0,PHO,104,0,2025,2025-01-07,True
4,240.0,48.0,99.0,0.485,11.0,35.0,0.314,12.0,19.0,0.632,...,4.0,25.0,24.2,1.0,PHI,102,1,2025,2025-03-03,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7007,240.0,47.0,89.0,0.528,14.0,31.0,0.452,19.0,25.0,0.760,...,5.0,40.0,31.4,6.0,DEN,126,0,2023,2022-12-28,True
7008,240.0,43.0,87.0,0.494,10.0,27.0,0.370,21.0,27.0,0.778,...,5.0,29.0,23.5,7.0,SAS,110,1,2024,2023-11-10,True
7009,240.0,43.0,92.0,0.467,12.0,34.0,0.353,12.0,19.0,0.632,...,6.0,29.0,20.7,11.0,MIN,117,0,2024,2023-11-10,False
7010,240.0,42.0,82.0,0.512,12.0,27.0,0.444,12.0,19.0,0.632,...,3.0,26.0,22.3,19.0,TOR,121,1,2024,2023-11-24,False


In [21]:
games_df.to_csv("nba_games.csv")